# 02 — 1D CNN Classifier

Train a 1D convolutional neural network on continuum-normalized APOGEE spectra
for single-epoch binary star detection. The CNN learns directly from the flux
array, avoiding hand-crafted feature engineering.

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import roc_auc_score

# path setup
os.chdir(os.path.join(os.path.dirname(os.path.abspath(".")), ".."))
sys.path.insert(0, os.getcwd())

from src.models import BinaryStarCNN, create_data_loaders, train_cnn
from src.utils import plot_roc_pr, plot_confusion_matrix, print_metrics, save_figure
from config import MODEL_CONFIG, VIS_CONFIG, FEATURE_CONFIG

%matplotlib inline
plt.rcParams["figure.dpi"] = VIS_CONFIG["dpi"]

# select best available device
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

## 1. Load Spectra

In [ ]:
# load preprocessed spectra and labels
spectra = np.load("data/spectra_matrix.npy")
labels = np.load("data/spectra_labels.npy")

print(f"Spectra shape: {spectra.shape}  (n_stars, n_pixels)")
print(f"Labels shape:  {labels.shape}")
print(f"\nClass balance:")
print(f"  Single (0): {(labels == 0).sum()}  ({(labels == 0).mean():.1%})")
print(f"  Binary (1): {(labels == 1).sum()}  ({(labels == 1).mean():.1%})")

# filter out rows with NaN pixels
nan_mask = np.isnan(spectra).any(axis=1)
n_nan_rows = nan_mask.sum()
print(f"\nRows with NaN pixels: {n_nan_rows}")
if n_nan_rows > 0:
    spectra = spectra[~nan_mask]
    labels = labels[~nan_mask]
    print(f"After filtering: {spectra.shape[0]} spectra remain")

## 2. Create DataLoaders

In [ ]:
# create_data_loaders handles stratified train/val/test splitting internally
# training set uses noise augmentation via per-pixel error arrays
train_loader, val_loader, test_loader = create_data_loaders(spectra, labels)

print(f"Train batches: {len(train_loader)}  ({len(train_loader.dataset)} spectra)")
print(f"Val batches:   {len(val_loader)}  ({len(val_loader.dataset)} spectra)")
print(f"Test batches:  {len(test_loader)}  ({len(test_loader.dataset)} spectra)")
print(f"\nBatch size: {MODEL_CONFIG['cnn_batch_size']}")
print(f"Note: training set uses noise augmentation at the per-pixel error level.")

## 3. Model Architecture

In [ ]:
# instantiate the CNN
model = BinaryStarCNN(n_pixels=8575)

# print parameter counts per layer
total_params = 0
for name, param in model.named_parameters():
    n = param.numel()
    total_params += n
    print(f"  {name:30s}  {str(list(param.shape)):20s}  {n:>10,}")
print(f"\n  {'TOTAL':30s}  {'':20s}  {total_params:>10,}")

# verify forward pass shape
dummy = torch.randn(2, 1, 8575)
out = model(dummy)
print(f"\nDummy forward pass: input {dummy.shape} -> output {out.shape}")

## 4. Train

In [ ]:
# train the CNN with early stopping on validation AUROC
history = train_cnn(model, train_loader, val_loader, device=device)

print(f"\nBest epoch: {history['best_epoch']}")
print(f"Best val AUROC: {max(history['val_auroc']):.4f}")

In [ ]:
# plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs = np.arange(1, len(history["train_loss"]) + 1)
best_ep = history["best_epoch"]

# loss curves
ax1.plot(epochs, history["train_loss"], label="Train loss")
ax1.plot(epochs, history["val_loss"], label="Val loss")
ax1.axvline(best_ep, color="gray", linestyle="--", alpha=0.7, label=f"Best epoch ({best_ep})")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss (BCE)")
ax1.set_title("Training and Validation Loss")
ax1.legend()

# AUROC curve
ax2.plot(epochs, history["val_auroc"], color="C2", label="Val AUROC")
ax2.axvline(best_ep, color="gray", linestyle="--", alpha=0.7, label=f"Best epoch ({best_ep})")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("AUROC")
ax2.set_title("Validation AUROC")
ax2.legend()

plt.tight_layout()
save_figure(fig, "cnn_training_curves")
plt.show()

## 5. Evaluate on Test Set

In [ ]:
# evaluate on test set
model.eval()
model = model.to(device)

all_probs = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        logits = model(X_batch).squeeze(-1)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(y_batch.numpy())

y_test = np.concatenate(all_labels)
y_probs = np.concatenate(all_probs)
y_pred = (y_probs >= 0.5).astype(int)

# classification report and AUROC
auroc = print_metrics(y_test, y_pred, y_probs)

# confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
plot_confusion_matrix(y_test, y_pred, ax=ax)
save_figure(fig, "cnn_confusion_matrix")
plt.show()

In [ ]:
# ROC and PR curves
fig = plot_roc_pr(y_test, {"1D CNN": y_probs})
save_figure(fig, "cnn_roc_pr")
plt.show()

## 6. Saliency Maps

Which spectral regions drive the CNN's predictions? We compute gradient-based
saliency by backpropagating the output w.r.t. the input pixels.

In [ ]:
# gradient-based saliency maps
model.eval()

# collect a batch of test spectra separated by class
binary_spectra = []
single_spectra = []

for X_batch, y_batch in test_loader:
    for i in range(len(y_batch)):
        if y_batch[i] == 1 and len(binary_spectra) < 100:
            binary_spectra.append(X_batch[i])
        elif y_batch[i] == 0 and len(single_spectra) < 100:
            single_spectra.append(X_batch[i])
    if len(binary_spectra) >= 100 and len(single_spectra) >= 100:
        break

def compute_saliency(spectra_list):
    """Compute mean absolute gradient saliency for a list of spectra."""
    batch = torch.stack(spectra_list).to(device)
    batch.requires_grad_(True)
    logits = model(batch).squeeze(-1)
    logits.sum().backward()
    saliency = batch.grad.abs().mean(dim=0).squeeze().cpu().numpy()
    return saliency

saliency_binary = compute_saliency(binary_spectra)
saliency_single = compute_saliency(single_spectra)

# plot saliency vs pixel index, with line regions marked
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# overlay a template spectrum for reference
template = torch.stack(single_spectra).mean(dim=0).squeeze().numpy()
ax1.plot(template, color="gray", alpha=0.5, linewidth=0.3, label="Mean single spectrum")
ax1.set_ylabel("Normalized Flux")
ax1.set_title("Reference: Mean Single-Star Spectrum")
ax1.legend()

# saliency comparison
ax2.plot(saliency_single, color="C0", alpha=0.6, linewidth=0.5, label="Single stars")
ax2.plot(saliency_binary, color="C3", alpha=0.6, linewidth=0.5, label="Binary stars")
ax2.set_xlabel("Pixel Index")
ax2.set_ylabel("Mean |gradient|")
ax2.set_title("CNN Saliency Maps by Class")
ax2.legend()

# mark key absorption line regions from FEATURE_CONFIG
colors = plt.cm.Set2(np.linspace(0, 1, len(FEATURE_CONFIG["line_regions"])))
for idx, (line_name, regions) in enumerate(FEATURE_CONFIG["line_regions"].items()):
    for wl_lo, wl_hi in regions:
        # approximate pixel from wavelength (APOGEE log-lambda grid: 15100-17000 A)
        pix_lo = int((wl_lo - 15100) / (17000 - 15100) * 8575)
        pix_hi = int((wl_hi - 15100) / (17000 - 15100) * 8575)
        pix_lo = max(0, min(pix_lo, 8574))
        pix_hi = max(0, min(pix_hi, 8574))
        ax2.axvspan(pix_lo, pix_hi, alpha=0.15, color=colors[idx], label=line_name)
    # avoid duplicate legend entries
    handles, labels = ax2.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax2.legend(by_label.values(), by_label.keys(), fontsize=7, ncol=3)

plt.tight_layout()
save_figure(fig, "cnn_saliency")
plt.show()

## 7. Save Model

In [ ]:
# save model state dict
os.makedirs("data", exist_ok=True)
torch.save(model.state_dict(), "data/cnn_model.pt")
print("Saved CNN model to data/cnn_model.pt")

# save test predictions for the comparison notebook
np.savez(
    "data/cnn_test_preds.npz",
    y_true=y_test,
    y_pred=y_pred,
    y_probs=y_probs,
)
print("Saved test predictions to data/cnn_test_preds.npz")